# 579. Find Cumulative Salary of an Employee

## Problem Description
The **Employee** table holds salary information for each employee by month.  

We need to:
- Calculate the **cumulative sum of salary over a period of 3 months**,  
- **Exclude the most recent month** for each employee,  
- Display results ordered by `Id` ascending and then by `Month` descending.

---

## Schema

### Table: Employee
| Column Name | Type    | Description                          |
|-------------|---------|--------------------------------------|
| Id          | INT     | Employee identifier                  |
| Month       | INT     | Month number (1–12)                  |
| Salary      | INT     | Salary for that month                |

**Primary Key:** `(Id, Month)`

---

## Sample Data

### Input: Employee
| Id | Month | Salary |
|----|-------|--------|
| 1  | 1     | 20     |
| 2  | 1     | 20     |
| 1  | 2     | 30     |
| 2  | 2     | 30     |
| 3  | 2     | 40     |
| 1  | 3     | 40     |
| 3  | 3     | 60     |
| 1  | 4     | 60     |
| 3  | 4     | 70     |

---

## Expected Output

| Id | Month | Salary |
|----|-------|--------|
| 1  | 3     | 90     |
| 1  | 2     | 50     |
| 1  | 1     | 20     |
| 2  | 1     | 20     |
| 3  | 3     | 100    |
| 3  | 2     | 40     |

---

## Explanation
- **Employee 1**:  
  - Exclude most recent month (4).  
  - Months considered: 3 (40), 2 (30), 1 (20).  
  - Cumulative sums:  
    - Month 3 → 40+30+20 = 90  
    - Month 2 → 30+20 = 50  
    - Month 1 → 20  

- **Employee 2**:  
  - Exclude most recent month (2).  
  - Only Month 1 remains → 20  

- **Employee 3**:  
  - Exclude most recent month (4).  
  - Months considered: 3 (60), 2 (40).  
  - Cumulative sums:  
    - Month 3 → 60+40 = 100  
    - Month 2 → 40  

---

## PySpark Code: Create DataFrame and Temp View

```python

In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType

# Schema for Employee
employee_schema = StructType([
    StructField("Id", IntegerType(), False),
    StructField("Month", IntegerType(), False),
    StructField("Salary", IntegerType(), False)
])

# Data for Employee
employee_data = [
    (1, 1, 20),
    (2, 1, 20),
    (1, 2, 30),
    (2, 2, 30),
    (3, 2, 40),
    (1, 3, 40),
    (3, 3, 60),
    (1, 4, 60),
    (3, 4, 70)
]

# Create DataFrame
employee_df = spark.createDataFrame(employee_data, employee_schema)

# Register Temp View
employee_df.createOrReplaceTempView("Employee")

# Quick check
employee_df.show()


In [0]:
%sql
with cte as (
    Select 
    id , month , salary
    ,row_number()over(partition by Id  order by Month  desc ) as rn 
    ,coalesce( lag(salary ,1 )over(partition by Id order by Month asc) ,0)as prev_sal
    ,coalesce( lag(salary ,2 )over(partition by Id order by Month asc) ,0)as prev_sal_2
    from Employee
)
Select id , month 
, (salary + prev_sal + prev_sal_2   ) as salary
from cte
where rn <> 1
order by id asc , month desc


Above gives correct result , but this code is not flwixible  to write for 100 cummulative 

In [0]:
%sql
with cte as (
    Select 
    id , month 
    ,row_number()over(partition by Id  order by Month  desc ) as rn 
   ,sum(salary)over(partition by Id order by Month asc rows between 2 preceding and current row)as salary
    from Employee
)
Select id , month  , salary
--, (salary + prev_sal + prev_sal_2   ) as salary
from cte
where rn <> 1
order by id asc , month desc

In [0]:
%sql
with cte as (
    Select 
    id , month 
    ,row_number()over(partition by Id  order by Month  desc ) as rn 
   ,sum(salary)over(partition by Id order by Month asc range   between 2 preceding and current row)as salary
    from Employee
)
Select id , month  , salary
--, (salary + prev_sal + prev_sal_2   ) as salary
from cte
where rn <> 1
order by id asc , month desc

difference between `ROWS` and `RANGE`.

---

## Example Data
Imagine this table ordered by `Month`:

| Month | Salary |
|-------|--------|
| 1     | 100    |
| 2     | 200    |
| 2     | 300    |
| 3     | 400    |

Notice: Month **2** appears twice.

---

## Using `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`

```sql
SUM(Salary) OVER (
  ORDER BY Month
  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
)
```

- For the **second row (Month=2, Salary=200)** → it looks at **exactly 2 rows before + current row**:
  - Row 1 (Month=1, Salary=100)
  - Row 2 (Month=2, Salary=200)
  → Sum = 300

- For the **third row (Month=2, Salary=300)** → it looks at **rows 1, 2, 3** (because they are the 2 preceding + current row):
  - 100 + 200 + 300 = 600

👉 **ROWS counts rows by position, not by value.**

---

## Using `RANGE BETWEEN 2 PRECEDING AND CURRENT ROW`

```sql
SUM(Salary) OVER (
  ORDER BY Month
  RANGE BETWEEN 2 PRECEDING AND CURRENT ROW
)
```

- For the **second row (Month=2, Salary=200)** → it looks at **all rows where Month is between (2‑2) and 2** → Months 0–2.
  - That includes **both rows with Month=2** (200 and 300).
  - So the sum = 100 + 200 + 300 = 600

- For the **third row (Month=2, Salary=300)** → same logic, Month=2, so it again includes **both rows with Month=2**.
  - Sum = 100 + 200 + 300 = 600

👉 **RANGE groups by the value of the ordering column. If duplicates exist, it includes all of them.**

---

## 🎯 Key Takeaway
- **ROWS** → “last N rows by position.”  
- **RANGE** → “all rows whose ordering column values fall within a numeric/date range.”  

For your **cumulative salary problem (last 3 months)**:
- You want “last 3 months by position,” not “all rows with Month values in a range.”  
- So **use `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`** — it’s predictable and works the same in SQL Server, Oracle, Databricks, etc.

---

Would you like me to show you your **LeetCode 579 query written with `ROWS`** so you can see exactly how it applies to that problem?